# 06 · From cold observations to event evidence

## Context

A threshold is useful when its meaning is explicit. Here, severe cold means an
observed PRISM daily minimum below −10 °C.

## Question

Where did severe cold persist for at least two days, and how synchronized was
its occurrence with the center of the study region?

## Analysis story

We move from temperature to states, from states to events, and from states to a
synchrony map. Each transition is one named verb.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

## Prepare · Keep the threshold next to its units

In [ ]:
from pathlib import Path

import xarray as xr

# Find the repository from either a root-level documentation build or a kernel
# started beside this notebook, then open the checksum-controlled PRISM extract.
data_path = next(
    candidate / "data" / "vignettes" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "data" / "vignettes" / "prism_boulder_january_2024.nc").exists()
)
prism = xr.open_dataset(data_path, engine="scipy").load()

# These assertions are part of the teaching contract: official source,
# canonical cube dimensions, complete daily time, and declared Celsius units.
assert prism.attrs["source"] == "PRISM Group, Oregon State University"
assert prism.attrs["is_synthetic"] == 0
assert prism.sizes == {"time": 30, "y": 24, "x": 24}
assert prism["tmax"].attrs["units"] == "degC"

cube = prism["tmin"]
assert cube.attrs["units"] == "degC"

## Pipes · Keep state, event, and synchrony questions separate

In [ ]:
from cubedynamics import pipe, verbs as v

severe_cold = (
    pipe(cube)
    | v.threshold_state(threshold=-10.0, direction="below", name="severe_cold")
).unwrap()

events = (pipe(severe_cold) | v.detect_events(min_duration=2)).unwrap()

synchrony = (
    pipe(severe_cold)
    | v.occurrence_synchrony(spatial_mode="reference", reference="center")
).unwrap()

assert len(events.catalog) > 0

## Figure · Follow the evidence from values to events

In [ ]:
import matplotlib.pyplot as plt

event_count = events.dataset["event_active"].sum("time")
reference_sync = synchrony["occurrence_synchrony"].isel(time_window_end=0)

fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
cube.isel(time=15).plot(ax=axes[0, 0], cmap="coolwarm", cbar_kwargs={"label": "°C"})
axes[0, 0].set_title("Observed minimum temperature · 16 January")
severe_cold["state"].isel(time=15).plot(ax=axes[0, 1], cmap="Blues", add_colorbar=False)
axes[0, 1].set_title("Below −10 °C state")
event_count.plot(ax=axes[1, 0], cmap="magma", cbar_kwargs={"label": "Event days"})
axes[1, 0].set_title("Days retained in ≥2-day events")
reference_sync.plot(ax=axes[1, 1], cmap="viridis", vmin=0, vmax=1)
axes[1, 1].set_title("Occurrence synchrony with center cell")
plt.show()

## What the figure tells us

The threshold isolates the observed mid-January outbreak. Duration filtering
distinguishes persistence, while synchrony shows where its timing matched the
region's center.

## Try the next variation

Change only the threshold to −15 °C and compare retained events.